# pairadigm Example

This notebook contains a worked exampe of using the `pairadigm` package with an open-source LLM via Ollama to measure hate speech using the Measuring Hate Speech dataset from Kennedy et. al. (2020) and Sachdeva et. al. (2022). 

- Kennedy, C. J., Bacon, G., Sahn, A., & von Vacano, C. (2020). Constructing interval variables via faceted Rasch measurement and multitask deep learning: a hate speech application. arXiv preprint arXiv:2009.10277.
- Pratik Sachdeva, Renata Barreto, Geoff Bacon, Alexander Sahn, Claudia von Vacano, and Chris Kennedy. 2022. The Measuring Hate Speech Corpus: Leveraging Rasch Measurement Theory for Data Perspectivism. In Proceedings of the 1st Workshop on Perspectivist Approaches to NLP @LREC2022, pages 83–94, Marseille, France. European Language Resources Association.

In [11]:
# !pip install pairadigm

In [12]:
import pandas as pd
import pairadigm as pdm
import dotenv
import os
import datasets 

In [13]:
# Load the data
dataset = datasets.load_dataset('ucberkeley-dlab/measuring-hate-speech')
df = dataset['train'].to_pandas()
df.describe()

,comment_id,annotator_id,platform,sentiment,respect,insult,humiliate,status,dehumanize,violence,...,hatespeech,hate_speech_score,infitms,outfitms,annotator_severity,std_err,annotator_infitms,annotator_outfitms,hypothesis,annotator_age
count,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135556.00000,135556.000000,135556.000000,135556.000000,135556.000000,...,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135451.000000
mean,23530.416138,5567.097812,1.281352,2.954307,2.828875,2.56331,2.278638,2.698575,1.846211,1.052045,...,0.744733,-0.567428,1.034322,1.001052,-0.018817,0.300588,1.007158,1.011841,0.014589,37.910772
std,12387.194125,3230.508937,1.023542,1.231552,1.309548,1.38983,1.370876,0.898500,1.402372,1.345706,...,0.932260,2.380003,0.496867,0.791943,0.487261,0.236380,0.269876,0.675863,0.613006,11.641276
min,1.000000,1.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,...,0.000000,-8.340000,0.100000,0.070000,-1.820000,0.020000,0.390000,0.280000,-1.578693,18.000000
25%,18148.000000,2719.000000,0.000000,2.000000,2.000000,2.00000,1.000000,2.000000,1.000000,0.000000,...,0.000000,-2.330000,0.710000,0.560000,-0.380000,0.030000,0.810000,0.670000,-0.341008,29.000000
50%,20052.000000,5602.500000,1.000000,3.000000,3.000000,3.00000,3.000000,3.000000,2.000000,0.000000,...,0.000000,-0.340000,0.960000,0.830000,-0.020000,0.340000,0.970000,0.850000,0.110405,35.000000
75%,32038.250000,8363.000000,2.000000,4.000000,4.000000,4.00000,3.000000,3.000000,3.000000,2.000000,...,2.000000,1.410000,1.300000,1.220000,0.350000,0.420000,1.170000,1.130000,0.449555,45.000000
max,50070.000000,11142.000000,3.000000,4.000000,4.000000,4.00000,4.000000,4.000000,4.000000,4.000000,...,2.000000,6.300000,5.900000,9.000000,1.360000,1.900000,2.010000,9.000000,0.987511,81.000000


In [14]:
# For exploration 
df.sort_values('comment_id')

,comment_id,annotator_id,platform,sentiment,respect,insult,humiliate,status,dehumanize,violence,...,annotator_religion_hindu,annotator_religion_jewish,annotator_religion_mormon,annotator_religion_muslim,annotator_religion_nothing,annotator_religion_other,annotator_sexuality_bisexual,annotator_sexuality_gay,annotator_sexuality_straight,annotator_sexuality_other
70119,1,712,0,4.0,4.0,4.0,3.0,4.0,3.0,0.0,...,False,False,False,False,False,False,False,False,True,False
50410,1,8185,0,3.0,3.0,3.0,3.0,3.0,3.0,0.0,...,False,False,False,False,False,False,False,True,False,False
1064,1,3330,0,4.0,3.0,3.0,3.0,3.0,2.0,1.0,...,False,False,False,False,True,False,False,False,True,False
23298,1,962,0,4.0,4.0,3.0,2.0,3.0,3.0,0.0,...,False,False,False,False,True,False,False,False,True,False
36676,2,4328,0,4.0,4.0,4.0,3.0,3.0,2.0,0.0,...,False,False,False,False,False,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55109,50068,3763,3,1.0,1.0,0.0,0.0,2.0,0.0,0.0,...,False,False,False,False,False,False,False,False,True,False
30185,50069,9083,3,4.0,4.0,3.0,3.0,3.0,3.0,1.0,...,False,False,False,False,False,True,True,False,False,False
85858,50070,7647,3,4.0,4.0,4.0,4.0,4.0,0.0,4.0,...,False,False,False,False,False,False,False,False,True,False
40975,50070,8656,3,4.0,4.0,4.0,3.0,4.0,3.0,4.0,...,False,False,False,False,True,False,False,False,True,False


In [15]:
# Pull relevant item data for example
cols = ['comment_id', 'text', 'hate_speech_score']
data = df[cols].drop_duplicates().copy()
data

,comment_id,text,hate_speech_score
0,47777,Yes indeed. She sort of reminds me of the elde...,-3.90
1,39773,The trans women reading this tweet right now i...,-6.52
2,47101,Question: These 4 broads who criticize America...,0.36
3,43625,It is about time for all illegals to go back t...,0.26
4,12538,For starters bend over the one in pink and kic...,1.54
...,...,...,...
135546,26287,🔥PUBG JAPAN SERIES 🔥Grade2 Day2 6/7 <Round8> 1...,-3.10
135547,14785,لا تتشمت الرجال مسكين يعاني كس امه 😂. يقول يال...,-0.20
135549,30588,"Umar khalid, sahela rashid, kanhiyan kumar जैस...",-1.69
135550,21008,@AbeShinzo @realDonaldTrump @shinzoabe 独裁者は行きま...,-2.49


In [16]:
# Pull a sample across bins of the hate speech score
data['bin'] = pd.cut(data['hate_speech_score'], 
                     bins=5, 
                     labels=['0-20', '20-40', '40-60', '60-80', '80-100'])

items_per_bin = 10 # Feel free to increase this is you're using a closed LLM API

sample = data.groupby('bin', observed=True).apply(lambda x: x.sample(items_per_bin, random_state=42), 
                                                  include_groups=False).reset_index(drop=True)
sample

,comment_id,text,hate_speech_score
0,11870,Got it. Thanks for the clarifying and congrats...,-5.65
1,22840,URL helping the rohingya refugees is also anot...,-5.60
2,10695,"That's why that one came here, went through th...",-5.79
3,32523,New Issue from the @GaDjournal on Humanitarian...,-6.21
4,5360,Fellow trans girl here! Love and support confi...,-8.34
5,20877,Wait— Dudes are clowning this kid and calling ...,-6.90
6,7153,"Such an amazing culture and religion, they're ...",-5.70
7,14407,"As a Pale white Algerian, I frequently play on...",-5.80
8,2320,That sounds epic!!! Almost All my friends at s...,-6.01
9,39121,Very sad to hear the news of the demise of res...,-5.85


# Build pairadigm

The central piece of how pairadigm works is the CGCoT prompting used. These prompts guide how the user intends for the LLM to understand the nuance required to accurately measure the construct from text. The goal is to take the comparison task for the LLM from a reasoning task - where it would have to come up with this thought process on its own - to a pattern recognition task. 

In [17]:
# Transparency: Prompts developed with assistance from Gemini Pro 3.1
cgcot_prompts = [
    'Summarize the following text: {text}', 
    'Does the text target a specific group or individual based on demographic characteristics (e.g., race, religion, gender, sexual orientation, disability)? If yes, explicitly AND CONCISELY identify the target. Text: {text}',
    'Does the text use explicit slurs, profanity, or profoundly insulting language specifically directed at the target? If yes, list the specific words or phrases. Text: {text}',
    'Does the text compare the target to non-human entities (e.g., animals, pests, diseases, dirt) or imply they are biologically/morally inferior? BRIEFLY and CONCISELY explain how. Text: {text}',
    'Does the text express a desire for the target to experience violence, discrimination, exclusion, or harm? Text: {text}'
]

In [18]:
p = pdm.Pairadigm(
    data=sample,
    item_id_name='comment_id', 
    text_name='text', 
    cgcot_prompts=cgcot_prompts,
    model_name='ministral-3:8b', # Feel free to swap in your model of choice here
    api_key=None, # If using a closed API, set this to your API key
    target_concept='hate speech'
)

# Review Clients
p.get_clients_info()

,index,model_name,provider
0,0,ministral-3:8b,ollama


In [19]:
# Let's make sure the clients can connect before we generate all the breakdowns and pairwise annotations
p.test_clients_connection(test_prompt='What is 2+2?')

Testing LLM client connections using the prompt 'What is 2+2?'...
Test response from ministral-3:8b: The sum of **2 + 2** is **4**.

### Breakdown:
- **Addition** is the process of combining two or more numbers to find their total.
- Here, **2** (the first addend) + **2** (the second addend) = **4** (the sum).

### Verification:
- **Counting**: If you have 2 apples and add 2 more, you now have 4 apples.
- **Number Line**: Starting at 2, moving 2 units forward lands you on 4.


{'ministral-3:8b': True}

The first step of the pairadigm process is generating the breakdowns of the items using the CGCoT prompts provided. This will run the answer to each prompt recursively through the LLM to build the breakdown, as discussed in Wu et. al.

In [ ]:
# Get CGCOT Breakdowns (this takes a while if using a local model)
p.generate_breakdowns()


Generating breakdowns for 50 items using: ministral-3:8b
[50/50] 100.0% complete
Completed: 50/50 items

Breakdowns added to [object name].data with column name(s): CGCoT_Breakdown


In [21]:
# Save so we don't have to do that again
p.save('hatespeech_example.pkl')
p.data.to_csv('hatespeech_example.csv', index=False)

Pairadigm object saved successfully to: hatespeech_example.pkl


Let's review some of the breakdowns.

In [22]:
# Review some breakdowns (min hate speech score in sample)
print(p.data.loc[p.data['hate_speech_score'] == min(p.data['hate_speech_score']), 'CGCoT_Breakdown'].values[0])

Original Text: Fellow trans girl here! Love and support confirmed!
Prompt 1 response: **Summary:**
The text is a brief, affirming message from a transgender girl to another, expressing love, support, and solidarity. It conveys unconditional acceptance and encouragement.
Prompt 2 response: **Target:** Transgender girls (specifically, likely cisgender women or allies identifying as such).
Prompt 3 response: No.
Prompt 4 response: No. The text expresses solidarity and affirmation without dehumanizing or implying inferiority.
Prompt 5 response: No. The text expresses solidarity, support, and love.


In [23]:
# Review some breakdowns (max hate speech score in sample)
print(p.data.loc[p.data['hate_speech_score'] == max(p.data['hate_speech_score']), 'CGCoT_Breakdown'].values[0])

Original Text: see this is why aliens wont come here well just call them weird and enslave them or kill them all
Prompt 1 response: The text humorously suggests that humanity’s history of exploitation, violence, and intolerance toward other groups (e.g., enslavement, genocide) implies that if extraterrestrial life were to exist, Earth would likely be deemed unworthy of contact due to its violent and discriminatory nature. The phrasing mocks human behavior by framing it as a reason aliens might avoid interaction.
Prompt 2 response: **Target:** **Generalized xenophobic/racist framing**—implicitly targets **non-white, non-Western, or "othered" groups** (e.g., racialized minorities, immigrants, or marginalized communities) via **alien metaphor** as a proxy for dehumanization. **Gender:** Likely **masculine/patriarchal** (aggressive, exclusionary language). **Religion/Sexual Orientation/Disability:** No explicit targeting.
Prompt 3 response: Yes.

- **"enslave them"**
- **"kill them all"**


As you can see if you used the default model, the breakdowns can be highly (sometimes overly) detailed. Users can control this in the CGCoT prompts if desired by prompting the model to be concise (or "CONCISE" if its stubborn).

In [24]:
# Pair items
p.generate_pairings(num_pairs_per_item=5,
                    make_splits=True, # Whether to make train/test splits or just generate pairs
                    breakdowns=True) # breakdowns=True let's the object know it should attach the generated breakdowns, otherwise it will just generate the pairs.

Item-level splits — train: 34, eval: 8, test: 8
Pairwise DataFrame with breakdowns created and stored in self.pairwise_df (item-level splits applied; see 'item1_split' / 'item2_split' columns)


/Users/mlchrzan/opt/anaconda3/envs/pairadigm/lib/python3.12/site-packages/pairadigm/core.py:1420: UserWarning: Item-level splits have been applied (train/eval/test). Pairs are generated within each split to prevent data leakage during model training. Please inspect your splits to confirm they are representative of your data before using them for model training.
  warnings.warn(


,item1,item2,item1_split,item2_split,breakdown1,breakdown2
0,5360,14407,train,train,Original Text: Fellow trans girl here! Love an...,"Original Text: As a Pale white Algerian, I fre..."
1,48374,4328,train,train,Original Text: Straight white Male but Fuckin ...,Original Text: NTA. You should tell your Dad w...
2,32319,14352,train,train,Original Text: No bitch apply enough pressure ...,"Original Text: Beat it out of her, or throw he..."
3,15892,5360,train,train,Original Text: Hope he gets PTSD when I rip Fa...,Original Text: Fellow trans girl here! Love an...
4,28090,4328,train,train,Original Text: @freetobe66 @michaelbeatty3 @1R...,Original Text: NTA. You should tell your Dad w...
...,...,...,...,...,...,...
142,45712,48705,test,test,Original Text: America God Said Stop Abortions...,Original Text: Bitch in the white had cotton m...
143,3068,1952,test,test,Original Text: MGTOW is fucking gay why don't ...,Original Text: this stupid bitch im gonna cloc...
144,27488,36407,test,test,Original Text: @r4ndie98 Gay ass cunt!\nPrompt...,"Original Text: No, he's giving arms to Saudi A..."
145,27488,48705,test,test,Original Text: @r4ndie98 Gay ass cunt!\nPrompt...,Original Text: Bitch in the white had cotton m...


In [25]:
# Generate LLM Pairwise Annotations (this also takes a while if using a local model)
p.generate_pairwise_annotations()

[default] Completed 50/147 comparisons
[default] Completed 100/147 comparisons


,item1,item2,item1_split,item2_split,breakdown1,breakdown2,decision,justification
0,5360,14407,train,train,Original Text: Fellow trans girl here! Love an...,"Original Text: As a Pale white Algerian, I fre...",Text2,FINAL ANSWER: **Description 2**\n\nJUSTIFICATI...
1,48374,4328,train,train,Original Text: Straight white Male but Fuckin ...,Original Text: NTA. You should tell your Dad w...,Text2,FINAL ANSWER: **Description 2**\n\nJUSTIFICATI...
2,32319,14352,train,train,Original Text: No bitch apply enough pressure ...,"Original Text: Beat it out of her, or throw he...",Text2,**FINAL ANSWER:** **Description 2**\n\n**JUSTI...
3,15892,5360,train,train,Original Text: Hope he gets PTSD when I rip Fa...,Original Text: Fellow trans girl here! Love an...,Text1,FINAL ANSWER: **Description 1**\n\nJUSTIFICATI...
4,28090,4328,train,train,Original Text: @freetobe66 @michaelbeatty3 @1R...,Original Text: NTA. You should tell your Dad w...,Text1,**FINAL ANSWER:** **Description 1**\n\n**JUSTI...
...,...,...,...,...,...,...,...,...
142,45712,48705,test,test,Original Text: America God Said Stop Abortions...,Original Text: Bitch in the white had cotton m...,Text2,FINAL ANSWER: **Description 2**\n\nJUSTIFICATI...
143,3068,1952,test,test,Original Text: MGTOW is fucking gay why don't ...,Original Text: this stupid bitch im gonna cloc...,Text2,FINAL ANSWER: **Description 2**\n\nJUSTIFICATI...
144,27488,36407,test,test,Original Text: @r4ndie98 Gay ass cunt!\nPrompt...,"Original Text: No, he's giving arms to Saudi A...",Text1,**FINAL ANSWER:** **Description 1**\n\n**JUSTI...
145,27488,48705,test,test,Original Text: @r4ndie98 Gay ass cunt!\nPrompt...,Original Text: Bitch in the white had cotton m...,Text1,**FINAL ANSWER:** **Description 1**\n\n**JUSTI...


In [26]:
# Fix decisions for the spots where the model correctly answered but didn't follow pairadigm's formatting instructions
p.pairwise_df['decision'] = p.pairwise_df['justification'].str.extract(r"FINAL ANSWER:\s*.*?(Description 1|Description 2|Tie).*?")

p.pairwise_df['decision'] = p.pairwise_df['decision'].replace({
    'Description 1': 'Text1',
    'Description 2': 'Text2'
})

p.pairwise_df

,item1,item2,item1_split,item2_split,breakdown1,breakdown2,decision,justification
0,5360,14407,train,train,Original Text: Fellow trans girl here! Love an...,"Original Text: As a Pale white Algerian, I fre...",Text2,FINAL ANSWER: **Description 2**\n\nJUSTIFICATI...
1,48374,4328,train,train,Original Text: Straight white Male but Fuckin ...,Original Text: NTA. You should tell your Dad w...,Text2,FINAL ANSWER: **Description 2**\n\nJUSTIFICATI...
2,32319,14352,train,train,Original Text: No bitch apply enough pressure ...,"Original Text: Beat it out of her, or throw he...",Text2,**FINAL ANSWER:** **Description 2**\n\n**JUSTI...
3,15892,5360,train,train,Original Text: Hope he gets PTSD when I rip Fa...,Original Text: Fellow trans girl here! Love an...,Text1,FINAL ANSWER: **Description 1**\n\nJUSTIFICATI...
4,28090,4328,train,train,Original Text: @freetobe66 @michaelbeatty3 @1R...,Original Text: NTA. You should tell your Dad w...,Text1,**FINAL ANSWER:** **Description 1**\n\n**JUSTI...
...,...,...,...,...,...,...,...,...
142,45712,48705,test,test,Original Text: America God Said Stop Abortions...,Original Text: Bitch in the white had cotton m...,Text2,FINAL ANSWER: **Description 2**\n\nJUSTIFICATI...
143,3068,1952,test,test,Original Text: MGTOW is fucking gay why don't ...,Original Text: this stupid bitch im gonna cloc...,Text2,FINAL ANSWER: **Description 2**\n\nJUSTIFICATI...
144,27488,36407,test,test,Original Text: @r4ndie98 Gay ass cunt!\nPrompt...,"Original Text: No, he's giving arms to Saudi A...",Text1,**FINAL ANSWER:** **Description 1**\n\n**JUSTI...
145,27488,48705,test,test,Original Text: @r4ndie98 Gay ass cunt!\nPrompt...,Original Text: Bitch in the white had cotton m...,Text1,**FINAL ANSWER:** **Description 1**\n\n**JUSTI...


In [27]:
# Save again
p.save('hatespeech_example.pkl')
p.pairwise_df.to_csv('hatespeech_pairwise_annotations.csv', index=False)

Pairadigm object saved successfully to: hatespeech_example.pkl


# Evaluate and Score

In [28]:
# Small check of rater consistency - are the pairwise annotations transitive?
p.check_transitivity()

{'decision': (1.0, 0, 69)}

In [29]:
p.score_items(normalization_scale=(0, 1))

[default [full]] Fitting Bradley-Terry model...
[default [split]] Fitting Bradley-Terry model...
[default full] Bradley-Terry model fitted with 147 comparisons
[default full] Mean hate speech score: 0.533
[default full] Std  hate speech score: 0.272
[default split] Bradley-Terry model fitted with 147 within-split comparisons
[default split] Mean hate speech score: 0.533
[default split] Std  hate speech score: 0.272

Summary statistics (Bradley_Terry_Score_full):
Score range: 0.000 to 1.000
25th percentile: 0.339
50th percentile (median): 0.559
75th percentile: 0.782

Highest scoring item on hate speech (score: 1.000):
Beat it out of her, or throw her to the curb. No use for uppity niggers.

Lowest scoring item on hate speech (score: 0.000):
Got it. Thanks for the clarifying and congrats for you on being open and proud about your sexuality. It's a hard journey for many to accept and is a beautiful thing when people can be open and confident about who they truly are!
  mean: 0.533
  medi

,comment_id,text,hate_speech_score,CGCoT_Breakdown,Bradley_Terry_Score_full,Bradley_Terry_Score_split
0,11870,Got it. Thanks for the clarifying and congrats...,-5.65,Original Text: Got it. Thanks for the clarifyi...,0.000000,0.000000
1,22840,URL helping the rohingya refugees is also anot...,-5.60,Original Text: URL helping the rohingya refuge...,0.325034,0.325034
2,10695,"That's why that one came here, went through th...",-5.79,"Original Text: That's why that one came here, ...",0.219850,0.219850
3,32523,New Issue from the @GaDjournal on Humanitarian...,-6.21,Original Text: New Issue from the @GaDjournal ...,0.145640,0.145640
4,5360,Fellow trans girl here! Love and support confi...,-8.34,Original Text: Fellow trans girl here! Love an...,0.140777,0.140777
5,20877,Wait— Dudes are clowning this kid and calling ...,-6.90,Original Text: Wait— Dudes are clowning this k...,0.241041,0.241041
6,7153,"Such an amazing culture and religion, they're ...",-5.70,Original Text: Such an amazing culture and rel...,0.494203,0.494203
7,14407,"As a Pale white Algerian, I frequently play on...",-5.80,"Original Text: As a Pale white Algerian, I fre...",0.266424,0.266424
8,2320,That sounds epic!!! Almost All my friends at s...,-6.01,Original Text: That sounds epic!!! Almost All ...,0.374539,0.374539
9,39121,Very sad to hear the news of the demise of res...,-5.85,Original Text: Very sad to hear the news of th...,0.393027,0.393027


In [30]:
# You know the deal by this point
p.save('hatespeech_example.pkl')
p.scored_df.to_csv('hatespeech_scored_items.csv', index=False)

Pairadigm object saved successfully to: hatespeech_example.pkl


In [33]:
p.plot_score_distribution(score_col='Bradley_Terry_Score_full')